# İspanya Gün Öncesi Elektrik Fiyat Tahmini — tam pipeline

Bu defter **ham CSV’den kilitli modele** kadar her şeyi çalıştırır.

Gerekli dosyalar yalnızca:
- `energy_dataset.csv`
- `weather_features.csv`

(kökte veya `data/raw/` altında). `src/`, `reports/` veya işlenmiş parquet **zorunlu değildir**.

| Kilit | Değer |
|---|---|
| Model | Ridge, kapalı form NumPy |
| α | **0.001** (walk-forward ile seçilir, test kullanılmaz) |
| Düzeltme | **METHOD_B** + expanding-historical addend |
| 24s üretim | **HAZIR DEĞİL** |

Korumalı `data/processed/` dosyalarına **yazılmaz**. Test seti seçim/ayar için kullanılmaz; yalnızca en sonda bir kez skorlanır.

Nedensellik iddiası yoktur.


## 0. Kurulum

```bash
pip install numpy pandas pyarrow scikit-learn matplotlib
jupyter notebook day_ahead_price_prediction.ipynb
```

Ağaç modelleri (HGB, varsa LightGBM/XGBoost) walk-forward’da birkaç dakika sürebilir. İlk tam koşu: hücreleri sırayla Run All.


In [ ]:
from __future__ import annotations

import math
from importlib.util import find_spec
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})

ROOT = Path.cwd()
for cand in (ROOT, Path("/Users/ibrahim/Desktop/elec_s"), *list(ROOT.parents)[:4]):
    if (cand / "energy_dataset.csv").exists() or (cand / "data" / "raw" / "energy_dataset.csv").exists():
        ROOT = cand
        break


def find_csv(name: str) -> Path:
    for p in (ROOT / name, ROOT / "data" / "raw" / name):
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} bulunamadı. Kök veya data/raw/ altına koyun.")


print("ROOT =", ROOT)
print("energy:", find_csv("energy_dataset.csv"))
print("weather:", find_csv("weather_features.csv"))


## 1. Sabitler ve sızıntısız yardımcılar


In [ ]:
TARGET = "price day ahead"
ID = "timestamp_utc"
EXPECTED_ROWS = 35064
LOCAL_TZ = "Europe/Madrid"
ALPHA_LOCKED = 0.001
TAU_HOURS = 720.0
TRAIN_FRAC, VAL_END_FRAC = 0.70, 0.85
TRAIN_CUTS = (0.50, 0.60, 0.70, 0.80)
RANDOM_STATE = 42
RUN_TREE_MODELS = True  # walk-forward HGB / LightGBM / XGBoost (birkaç dakika)
CITIES = ("Barcelona", "Bilbao", "Madrid", "Seville", "Valencia")
EMPTY_ENERGY = (
    "generation hydro pumped storage aggregated",
    "forecast wind offshore eday ahead",
)
NUMERIC_WEATHER = (
    "temp", "temp_min", "temp_max", "pressure", "humidity",
    "wind_speed", "wind_deg", "rain_1h", "rain_3h", "snow_3h", "clouds_all",
)
CAT_WEATHER = ("weather_id", "weather_main", "weather_description", "weather_icon")
GENERATION_LAG_SOURCES = (
    "generation biomass", "generation fossil brown coal/lignite", "generation fossil gas",
    "generation fossil hard coal", "generation fossil oil",
    "generation hydro pumped storage consumption", "generation hydro run-of-river and poundage",
    "generation hydro water reservoir", "generation nuclear", "generation other",
    "generation other renewable", "generation solar", "generation waste", "generation wind onshore",
)
RENEWABLE_GENERATION = (
    "generation biomass", "generation hydro run-of-river and poundage",
    "generation hydro water reservoir", "generation other renewable",
    "generation solar", "generation wind onshore",
)
FOSSIL_GENERATION = (
    "generation fossil brown coal/lignite", "generation fossil gas",
    "generation fossil hard coal", "generation fossil oil",
)
TOTAL_PRODUCTION = (
    "generation biomass", "generation fossil brown coal/lignite", "generation fossil gas",
    "generation fossil hard coal", "generation fossil oil",
    "generation hydro run-of-river and poundage", "generation hydro water reservoir",
    "generation nuclear", "generation other", "generation other renewable",
    "generation solar", "generation waste", "generation wind onshore",
)
WEATHER_LAG_FIELDS = ("temp", "humidity", "pressure", "wind_speed", "clouds_all", "rain_1h", "rain_3h", "snow_3h")
FRAC_COLS = (
    "fraction_high_price_last_7d",
    "fraction_high_price_last_14d",
    "fraction_high_price_last_30d",
)
FRAC_WINDOWS = {FRAC_COLS[0]: 168, FRAC_COLS[1]: 336, FRAC_COLS[2]: 720}


def sanitize(name: str) -> str:
    return name.strip().lower().replace("/", "_").replace("-", "_").replace(" ", "_")


def safe_divide(num: pd.Series, den: pd.Series) -> pd.Series:
    n = num.to_numpy(dtype=float)
    d = den.to_numpy(dtype=float)
    out = np.full(len(n), np.nan)
    ok = np.isfinite(d) & (d > 0) & np.isfinite(n)
    out[ok] = n[ok] / d[ok]
    return pd.Series(out, index=num.index)


def lag_hours(s: pd.Series, h: int) -> pd.Series:
    if h <= 0:
        raise ValueError("yalnızca pozitif gecikme")
    return s.shift(h)


def circular_mean_degrees(values: pd.Series) -> float:
    arr = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    rad = np.deg2rad(arr)
    return float(np.rad2deg(np.arctan2(np.mean(np.sin(rad)), np.mean(np.cos(rad)))) % 360)


def deterministic_mode(values: pd.Series):
    cleaned = values.dropna()
    if cleaned.empty:
        return np.nan
    as_str = cleaned.astype(str)
    counts = as_str.value_counts()
    winner = sorted(counts[counts == counts.max()].index.tolist())[0]
    return cleaned.loc[as_str == winner].iloc[0]


def smape(y, p) -> float:
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    denom = np.abs(y) + np.abs(p)
    out = np.zeros_like(y)
    ok = denom > 0
    out[ok] = 2.0 * np.abs(p[ok] - y[ok]) / denom[ok]
    return float(np.mean(out) * 100)


def metrics(y, p) -> dict:
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    err = p - y
    return {
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "R2": float(1 - np.sum(err ** 2) / np.sum((y - y.mean()) ** 2)),
        "sMAPE": smape(y, p),
        "bias": float(np.mean(err)),
    }


class Preprocessor:
    def fit(self, x: pd.DataFrame):
        arr = x.to_numpy(dtype=float)
        self.medians_ = np.nanmedian(arr, axis=0)
        filled = self._impute(arr)
        self.mean_ = filled.mean(axis=0)
        scale = filled.std(axis=0, ddof=0)
        scale[scale == 0] = 1.0
        self.scale_ = scale
        return self

    def _impute(self, arr: np.ndarray) -> np.ndarray:
        out = arr.copy()
        for j, med in enumerate(self.medians_):
            mask = np.isnan(out[:, j])
            if mask.any():
                out[mask, j] = 0.0 if not np.isfinite(med) else float(med)
        return out

    def transform(self, x: pd.DataFrame) -> np.ndarray:
        filled = self._impute(x.to_numpy(dtype=float))
        return (filled - self.mean_) / self.scale_


def ridge_predict(x_tr, y_tr, x_te, alpha: float):
    y_mean = float(np.mean(y_tr))
    yc = y_tr - y_mean
    xtx = x_tr.T @ x_tr + float(alpha) * np.eye(x_tr.shape[1])
    w = np.linalg.solve(xtx, x_tr.T @ yc)
    return x_te @ w + y_mean, w, y_mean


def expanding_addend(train_ts, resid_train) -> float:
    ts = pd.to_datetime(train_ts, utc=True)
    hours = ((ts - ts.max()) / pd.Timedelta(hours=1)).to_numpy(dtype=float)
    w = np.exp(hours / TAU_HOURS)
    if not np.isfinite(w).all() or w.sum() <= 0:
        return float(-np.mean(resid_train))
    return float(-np.average(resid_train, weights=w))


def causal_high_fraction(y: pd.Series, threshold: float, window: int) -> pd.Series:
    hist = y.shift(24)
    flag = pd.Series(np.where(hist.isna(), np.nan, (hist.to_numpy(dtype=float) > threshold).astype(float)), index=y.index)
    return flag.rolling(window, min_periods=window).mean()


def add_method_b(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    out = df.copy()
    y = out[TARGET]
    out[FRAC_COLS[0]] = causal_high_fraction(y, threshold, 168)
    out[FRAC_COLS[1]] = causal_high_fraction(y, threshold, 336)
    out[FRAC_COLS[2]] = causal_high_fraction(y, threshold, 720)
    return out


print("yardımcılar hazır")


## 2. Veri yükleme, temizlik, birleştirme


In [ ]:
def normalize_ts(df: pd.DataFrame, col: str) -> pd.DataFrame:
    out = df.copy()
    out[ID] = pd.to_datetime(out[col], utc=True, errors="coerce")
    if out[ID].dt.tz is None:
        raise ValueError("naive timestamp")
    return out


def _nan_runs(mask: np.ndarray):
    runs, i, n = [], 0, len(mask)
    while i < n:
        if mask[i]:
            j = i + 1
            while j < n and mask[j]:
                j += 1
            runs.append((i, j))
            i = j
        else:
            i += 1
    return runs


def interpolate_short_gaps(series: pd.Series, max_gap: int = 3):
    s = series.copy()
    original_na = s.isna()
    if int(original_na.sum()) == 0:
        return s
    interpolated = s.interpolate(method="time", limit_area="inside")
    fill_mask = np.zeros(len(s), dtype=bool)
    for start, end in _nan_runs(original_na.to_numpy()):
        is_edge = start == 0 or end == len(s)
        if (not is_edge) and (end - start) <= max_gap:
            fill_mask[start:end] = True
    s.iloc[fill_mask] = interpolated.iloc[fill_mask]
    return s


energy = normalize_ts(pd.read_csv(find_csv("energy_dataset.csv")), "time")
weather = pd.read_csv(find_csv("weather_features.csv"))
weather["city_name"] = weather["city_name"].astype(str).str.strip()
weather = normalize_ts(weather, "dt_iso")
print("ham energy", energy.shape, "ham weather", weather.shape)

energy = energy.sort_values(ID).reset_index(drop=True)
energy = energy.drop(columns=[c for c in EMPTY_ENERGY if c in energy.columns])
indexed = energy.set_index(ID, drop=False)
imputable = [c for c in energy.columns if (c.startswith("generation ") or c == "total load actual") and c not in (TARGET, "price actual")]
for col in imputable:
    indexed[col] = interpolate_short_gaps(indexed[col])
energy = indexed.reset_index(drop=True)
assert not energy[TARGET].isna().any()
print("hedef min/max", float(energy[TARGET].min()), float(energy[TARGET].max()))

keys = [ID, "city_name"]
keep = keys + list(NUMERIC_WEATHER) + list(CAT_WEATHER)
sizes = weather.groupby(keys, sort=False).size()
dup_index = sizes[sizes > 1].index
key_index = pd.MultiIndex.from_frame(weather[keys])
is_dup = key_index.isin(dup_index) if len(dup_index) else np.zeros(len(weather), dtype=bool)
unique_part = weather.loc[~is_dup, keep].copy()
if is_dup.any():
    grouped = weather.loc[is_dup, keep].groupby(keys, sort=True, dropna=False)
    agg = grouped.agg(
        temp=("temp", "mean"), temp_min=("temp_min", "mean"), temp_max=("temp_max", "mean"),
        pressure=("pressure", "median"), humidity=("humidity", "mean"),
        wind_speed=("wind_speed", "mean"), wind_deg=("wind_deg", circular_mean_degrees),
        rain_1h=("rain_1h", "max"), rain_3h=("rain_3h", "max"), snow_3h=("snow_3h", "max"),
        clouds_all=("clouds_all", "mean"),
        weather_id=("weather_id", deterministic_mode),
        weather_main=("weather_main", deterministic_mode),
        weather_description=("weather_description", deterministic_mode),
        weather_icon=("weather_icon", deterministic_mode),
    ).reset_index()
    weather_agg = pd.concat([unique_part, agg], ignore_index=True)
else:
    weather_agg = unique_part
weather_agg = weather_agg.sort_values(keys).reset_index(drop=True)

pieces = []
for col in list(NUMERIC_WEATHER) + list(CAT_WEATHER):
    wide = weather_agg.pivot(index=ID, columns="city_name", values=col).reindex(columns=list(CITIES))
    wide.columns = [f"{col}_{city}" for city in wide.columns]
    pieces.append(wide)
weather_wide = pd.concat(pieces, axis=1).sort_index().reset_index()

merged = energy.merge(weather_wide, on=ID, how="left", validate="one_to_one").sort_values(ID).reset_index(drop=True)
assert len(merged) == EXPECTED_ROWS
assert str(merged[ID].dt.tz) == "UTC"
print("birleşik", merged.shape, merged[ID].min(), "→", merged[ID].max())
print("price actual özellik olarak kullanılmayacak; kaynakta duruyor:", "price actual" in merged.columns)


## 3. Sızıntısız özellik mühendisliği (184 SAFE)


In [ ]:
def add_calendar(ts: pd.Series) -> pd.DataFrame:
    local = ts.dt.tz_convert(LOCAL_TZ)
    hour, dow, month, doy = local.dt.hour, local.dt.dayofweek, local.dt.month, local.dt.dayofyear
    two_pi = 2 * math.pi
    out = pd.DataFrame({
        "hour": hour.astype(np.int16), "day_of_week": dow.astype(np.int16),
        "day_of_month": local.dt.day.astype(np.int16), "month": month.astype(np.int16),
        "quarter": local.dt.quarter.astype(np.int16), "day_of_year": doy.astype(np.int16),
        "week_of_year": local.dt.isocalendar().week.astype(np.int16),
        "is_weekend": (dow >= 5).astype(np.int8),
        "is_month_start": local.dt.is_month_start.astype(np.int8),
        "is_month_end": local.dt.is_month_end.astype(np.int8),
        "is_year_start": local.dt.is_year_start.astype(np.int8),
        "is_year_end": local.dt.is_year_end.astype(np.int8),
        "hour_sin": np.sin(two_pi * hour / 24), "hour_cos": np.cos(two_pi * hour / 24),
        "dow_sin": np.sin(two_pi * dow / 7), "dow_cos": np.cos(two_pi * dow / 7),
        "month_sin": np.sin(two_pi * month / 12), "month_cos": np.cos(two_pi * month / 12),
        "day_of_year_sin": np.sin(two_pi * doy / 365.25), "day_of_year_cos": np.cos(two_pi * doy / 365.25),
    }, index=ts.index)
    return out


def assemble_features(df: pd.DataFrame) -> pd.DataFrame:
    cal = add_calendar(df[ID])
    load_f, solar_f, wind_f = df["total load forecast"], df["forecast solar day ahead"], df["forecast wind onshore day ahead"]
    fc = pd.DataFrame({
        "total_load_forecast": load_f, "forecast_solar_day_ahead": solar_f,
        "forecast_wind_onshore_day_ahead": wind_f, "renewable_forecast_total": solar_f + wind_f,
        "forecast_wind_share_of_load": safe_divide(wind_f, load_f),
        "forecast_solar_share_of_load": safe_divide(solar_f, load_f),
    }, index=df.index)
    price = df[TARGET]
    lag24, lag48, lag168 = lag_hours(price, 24), lag_hours(price, 48), lag_hours(price, 168)
    pair = pd.concat([lag24, lag48], axis=1)
    triple = pd.concat([lag24, lag48, lag168], axis=1)
    tgt = pd.DataFrame({
        "price_day_ahead_lag_24": lag24, "price_day_ahead_lag_48": lag48, "price_day_ahead_lag_168": lag168,
        "price_mean_lag24_lag48": pair.mean(axis=1, skipna=False),
        "price_mean_lag24_lag48_lag168": triple.mean(axis=1, skipna=False),
        "price_std_lag24_lag48_lag168": triple.std(axis=1, ddof=0, skipna=False),
        "price_min_lag24_lag48_lag168": triple.min(axis=1, skipna=False),
        "price_max_lag24_lag48_lag168": triple.max(axis=1, skipna=False),
    }, index=df.index)
    load = pd.DataFrame(index=df.index)
    for h in (24, 48, 168):
        load[f"total_load_actual_lag_{h}"] = lag_hours(df["total load actual"], h)
    load["load_forecast_error_lag_24"] = lag_hours(df["total load actual"] - df["total load forecast"], 24)
    gen = pd.DataFrame(index=df.index)
    for col in GENERATION_LAG_SOURCES:
        base = sanitize(col)
        for h in (24, 168):
            gen[f"{base}_lag_{h}"] = lag_hours(df[col], h)
    total = df[list(TOTAL_PRODUCTION)].sum(axis=1, min_count=1)
    renewable = df[list(RENEWABLE_GENERATION)].sum(axis=1, min_count=1)
    fossil = df[list(FOSSIL_GENERATION)].sum(axis=1, min_count=1)
    share = safe_divide(renewable, total)
    for h in (24, 168):
        gen[f"total_generation_lag_{h}"] = lag_hours(total, h)
        gen[f"renewable_generation_lag_{h}"] = lag_hours(renewable, h)
        gen[f"fossil_generation_lag_{h}"] = lag_hours(fossil, h)
        gen[f"renewable_share_lag_{h}"] = lag_hours(share, h)
    wcols = {}
    for city in CITIES:
        ck = city.lower()
        rad = np.deg2rad(df[f"wind_deg_{city}"].to_numpy(dtype=float))
        wsin, wcos = pd.Series(np.sin(rad), index=df.index), pd.Series(np.cos(rad), index=df.index)
        for h in (24, 168):
            wcols[f"wind_deg_sin_{ck}_lag_{h}"] = lag_hours(wsin, h)
            wcols[f"wind_deg_cos_{ck}_lag_{h}"] = lag_hours(wcos, h)
        for field in WEATHER_LAG_FIELDS:
            for h in (24, 168):
                wcols[f"{field}_{ck}_lag_{h}"] = lag_hours(df[f"{field}_{city}"], h)
    national = {
        "temp_national_mean": df[[f"temp_{c}" for c in CITIES]].mean(axis=1),
        "humidity_national_mean": df[[f"humidity_{c}" for c in CITIES]].mean(axis=1),
        "wind_speed_national_mean": df[[f"wind_speed_{c}" for c in CITIES]].mean(axis=1),
        "clouds_all_national_mean": df[[f"clouds_all_{c}" for c in CITIES]].mean(axis=1),
        "rain_1h_national_max": df[[f"rain_1h_{c}" for c in CITIES]].max(axis=1),
    }
    for base, series in national.items():
        for h in (24, 168):
            wcols[f"{base}_lag_{h}"] = lag_hours(series, h)
    weather_hist = pd.DataFrame(wcols, index=df.index)
    feats = pd.concat([pd.DataFrame({ID: df[ID]}), cal, fc, tgt, load, gen, weather_hist], axis=1)
    x_cols = [c for c in feats.columns if c != ID]
    assert TARGET not in feats.columns and "price actual" not in feats.columns
    assert not any(c.endswith("_lag_1") for c in x_cols)
    assert len(x_cols) == 184, len(x_cols)
    return feats


features = assemble_features(merged)
x_cols = [c for c in features.columns if c != ID]
model = features.merge(merged[[ID, TARGET]], on=ID, how="left", validate="one_to_one")
print("SAFE özellik", len(x_cols), "satır", len(model))
print("sızıntı kontrolü: hedef ve price actual X'te yok")


## 4. Kronolojik 70 / 15 / 15 bölme (shuffle yok)


In [ ]:
n = len(model)
train_end = int(n * TRAIN_FRAC)
val_end = int(n * VAL_END_FRAC)
train = model.iloc[:train_end].copy()
val = model.iloc[train_end:val_end].copy()
test = model.iloc[val_end:].copy()
assert train[ID].max() < val[ID].min() < test[ID].min() or (val[ID].max() < test[ID].min())
assert train[ID].max() < val[ID].min()
assert val[ID].max() < test[ID].min()
dev = pd.concat([train, val], ignore_index=True).sort_values(ID).reset_index(drop=True)
print(f"train {len(train)}  val {len(val)}  test {len(test)}  dev {len(dev)}")
for name, df in ("train", train), ("val", val), ("test", test):
    print(f"  {name}: {df[ID].min()} → {df[ID].max()}  y_mean={df[TARGET].mean():.2f}")
print("TEST bundan sonra yalnızca kilitli skor için açılacak; seçim yok.")


## 5. Baseline (yalnızca validation)

Naive Lag-24, Ridge (α=0.1, bu aşamanın ızgarası), HistGradientBoosting.
Test bu hücrede **yok**.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

y_va = val[TARGET].to_numpy(dtype=float)
naive_va = val["price_day_ahead_lag_24"].to_numpy(dtype=float)
print("Naive Lag-24 validation", metrics(y_va, naive_va))

prep_b = Preprocessor().fit(train[x_cols])
ridge_va, _, _ = ridge_predict(prep_b.transform(train[x_cols]), train[TARGET].to_numpy(dtype=float), prep_b.transform(val[x_cols]), 0.1)
print("Ridge α=0.1 validation", metrics(y_va, ridge_va))

hgb = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=31, random_state=RANDOM_STATE, early_stopping=False)
hgb.fit(train[x_cols], train[TARGET])
print("HGB validation", metrics(y_va, hgb.predict(val[x_cols])))


## 6. Walk-forward (TRAIN+VALIDATION, 4 kat)

Test yüklenmez. Ridge α ızgarası + Naive + HGB (+ LightGBM/XGBoost varsa).


In [ ]:
def make_folds(df: pd.DataFrame):
    n = len(df)
    cuts = [int(n * f) for f in TRAIN_CUTS] + [n]
    folds = []
    for i, (a, b) in enumerate(zip(cuts[:-1], cuts[1:]), 1):
        tr, va = df.iloc[:a], df.iloc[a:b]
        assert tr[ID].max() < va[ID].min()
        folds.append((i, tr, va))
    return folds


folds = make_folds(dev)
rows = []


def eval_fold(name, y, p, fold):
    m = metrics(y, p)
    m.update(model=name, fold=fold)
    rows.append(m)
    return m


RIDGE_GRID = (0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0)

for i, tr, va in folds:
    yv = va[TARGET].to_numpy(dtype=float)
    eval_fold("Naive Lag-24", yv, va["price_day_ahead_lag_24"].to_numpy(dtype=float), i)
    prep = Preprocessor().fit(tr[x_cols])
    xtr, xva = prep.transform(tr[x_cols]), prep.transform(va[x_cols])
    ytr = tr[TARGET].to_numpy(dtype=float)
    for a in RIDGE_GRID:
        pred, _, _ = ridge_predict(xtr, ytr, xva, a)
        eval_fold(f"Ridge_a{a}", yv, pred, i)
    if RUN_TREE_MODELS:
        h = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=31, random_state=RANDOM_STATE, early_stopping=False)
        h.fit(tr[x_cols], tr[TARGET])
        eval_fold("HistGradientBoosting", yv, h.predict(va[x_cols]), i)
        if find_spec("lightgbm"):
            from lightgbm import LGBMRegressor
            m = LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=RANDOM_STATE, n_jobs=1, verbose=-1)
            m.fit(tr[x_cols], tr[TARGET])
            eval_fold("LightGBM", yv, m.predict(va[x_cols]), i)
        if find_spec("xgboost"):
            from xgboost import XGBRegressor
            m = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=RANDOM_STATE, n_jobs=1, verbosity=0)
            m.fit(tr[x_cols], tr[TARGET])
            eval_fold("XGBoost", yv, m.predict(va[x_cols]), i)
    print("fold", i, "bitti", "n_train", len(tr), "n_val", len(va))

wf = pd.DataFrame(rows)
summary = wf.groupby("model")[["MAE", "RMSE", "R2", "sMAPE", "bias"]].mean().sort_values("MAE")
display(summary)
best_ridge = summary.loc[summary.index.str.startswith("Ridge")].index[0]
print("walk-forward en iyi Ridge:", best_ridge)
print("Seçim kuralı (kilitli proje): α=0.001. Test kullanılmadı.")


## 7. METHOD_B (geliştirme katları)

α=0.001 sabit. Fold-train P75 ile nedensel 7/14/30g yüksek-fiyat oranları + expanding addend.
Test yok.


In [ ]:
method_rows = []
feat_plus = list(x_cols) + list(FRAC_COLS)
for i, tr, va in folds:
    p75 = float(np.quantile(tr[TARGET], 0.75))
    tr_b = add_method_b(tr, p75)
    va_b = add_method_b(pd.concat([tr, va], ignore_index=True), p75).iloc[len(tr):]
    prep = Preprocessor().fit(tr_b[feat_plus])
    ytr = tr_b[TARGET].to_numpy(dtype=float)
    pred_tr, _, _ = ridge_predict(prep.transform(tr_b[feat_plus]), ytr, prep.transform(tr_b[feat_plus]), ALPHA_LOCKED)
    pred_va, _, _ = ridge_predict(prep.transform(tr_b[feat_plus]), ytr, prep.transform(va_b[feat_plus]), ALPHA_LOCKED)
    addend = expanding_addend(tr_b[ID].to_numpy(), pred_tr - ytr)
    pred = pred_va + addend
    m = metrics(va[TARGET].to_numpy(dtype=float), pred)
    m.update(fold=i, p75=p75, addend=addend)
    method_rows.append(m)
    print(f"fold {i} METHOD_B MAE={m['MAE']:.4f} bias={m['bias']:.3f} addend={addend:.4f}")

mb = pd.DataFrame(method_rows)
print("METHOD_B mean MAE", mb["MAE"].mean(), "mean bias", mb["bias"].mean())
print("Yüksek fiyat sapması geliştirmede tamamen çözülmez; bu beklenen.")


## 8. Final model: geliştirmede fit, testte tek skor


In [ ]:
dev_p75 = float(np.quantile(dev[TARGET], 0.75))
dev_b = add_method_b(dev, dev_p75)
test_for_frac = pd.concat([dev, test], ignore_index=True)
test_b = add_method_b(test_for_frac, dev_p75).iloc[len(dev):].reset_index(drop=True)
# METHOD_B rolling uses only y shifted 24h; concatenating past y is required so windows are defined.
# Test *features other than METHOD_B fractions* come from test rows; P75 and preprocess are development-only.

prep_f = Preprocessor().fit(dev_b[feat_plus])
y_dev = dev_b[TARGET].to_numpy(dtype=float)
pred_dev, coef, intercept = ridge_predict(prep_f.transform(dev_b[feat_plus]), y_dev, prep_f.transform(dev_b[feat_plus]), ALPHA_LOCKED)
addend_final = expanding_addend(dev_b[ID].to_numpy(), pred_dev - y_dev)
pred_test_raw, _, _ = ridge_predict(prep_f.transform(dev_b[feat_plus]), y_dev, prep_f.transform(test_b[feat_plus]), ALPHA_LOCKED)
pred_test = pred_test_raw + addend_final
y_test = test[TARGET].to_numpy(dtype=float)
naive_test = test["price_day_ahead_lag_24"].to_numpy(dtype=float)

final_m = metrics(y_test, pred_test)
naive_m = metrics(y_test, naive_test)
tbl = pd.DataFrame({"Ridge+METHOD_B": final_m, "Naive Lag-24": naive_m})
display(tbl)
print("addend", addend_final, "dev P75", dev_p75)
print("MODEL_BEATS_NAIVE", final_m["MAE"] < naive_m["MAE"])

# yüksek fiyat dilimleri (rapor; yeniden kalibrasyon yok)
for q, lab in ((0.75, "P75+"), (0.90, "P90+"), (0.95, "P95+")):
    thr = float(np.quantile(y_dev, q))
    mask = y_test >= thr
    print(lab, "n=", int(mask.sum()), "bias=", float(np.mean(pred_test[mask] - y_test[mask])))


## 9. Test grafikleri (seçim yok, yalnızca kilitli skor)


In [ ]:
pred_df = pd.DataFrame({ID: test[ID].to_numpy(), "y_true": y_test, "y_pred": pred_test})
pred_df["residual"] = pred_df["y_pred"] - pred_df["y_true"]

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(pred_df[ID], pred_df["y_true"], lw=0.8, label="Gerçek")
axes[0].plot(pred_df[ID], pred_df["y_pred"], lw=0.8, alpha=0.85, label="Tahmin")
axes[0].legend(); axes[0].set_ylabel("€/MWh"); axes[0].set_title("Kilitli test")
axes[1].plot(pred_df[ID], pred_df["residual"], lw=0.7, color="C3")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_ylabel("Artık")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots()
ax.bar(["Naive Lag-24", "Ridge+METHOD_B"], [naive_m["MAE"], final_m["MAE"]])
ax.set_ylabel("Test MAE")
plt.tight_layout(); plt.show()

print("Geliştirme sapması walk-forward'da negatifti; testte işaret pozitife dönebilir.")
print("Bu, 'yüksek fiyat sorunu çözüldü' anlamına gelmez.")


## 10. Exact linear SHAP (geliştirme katları, TreeSHAP yok)

`φ = w ⊙ x_scaled`. `month` / `day_of_year` eşdoğrusal çifttir; tek sürücü gibi okunmaz.


In [ ]:
shap_abs = []
for i, tr, va in folds:
    p75 = float(np.quantile(tr[TARGET], 0.75))
    tr_b = add_method_b(tr, p75)
    va_b = add_method_b(pd.concat([tr, va], ignore_index=True), p75).iloc[len(tr):]
    prep = Preprocessor().fit(tr_b[feat_plus])
    ytr = tr_b[TARGET].to_numpy(dtype=float)
    xtr = prep.transform(tr_b[feat_plus])
    xva = prep.transform(va_b[feat_plus])
    _, w, _ = ridge_predict(xtr, ytr, xtr, ALPHA_LOCKED)
    phi = xva * w
    shap_abs.append(np.mean(np.abs(phi), axis=0))

mean_abs = np.mean(np.vstack(shap_abs), axis=0)
imp = pd.DataFrame({"feature": feat_plus, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False)
display(imp.head(15))
top = imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top["feature"], top["mean_abs_shap"])
ax.set_xlabel("ortalama |SHAP|")
ax.set_title("Linear SHAP — nedensellik değil")
plt.tight_layout(); plt.show()


## 11. 24 saatlik tahmin denetimi

D-1 ~12:00 CET kökeninde gün öncesi forecast yayın saati dosyada yok (UNKNOWN).
Akşam saatleri için `lag_24` gerçekleşmiş değerler henüz yok (FORBIDDEN).

STRICT: bu kolonlar sessiz doldurulmaz → **y_pred boş**. Üretime hazır değil.


In [ ]:
def group_of(name: str) -> str:
    cal_cols = {
        "hour", "day_of_week", "day_of_month", "month", "quarter", "day_of_year", "week_of_year",
        "is_weekend", "is_month_start", "is_month_end", "is_year_start", "is_year_end",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
        "day_of_year_sin", "day_of_year_cos",
    }
    fc = {
        "total_load_forecast", "forecast_solar_day_ahead", "forecast_wind_onshore_day_ahead",
        "renewable_forecast_total", "forecast_wind_share_of_load", "forecast_solar_share_of_load",
    }
    if name in FRAC_COLS:
        return "method_b"
    if name in cal_cols:
        return "calendar"
    if name in fc:
        return "day_ahead_forecast"
    if name.startswith("price_"):
        return "historical_target"
    if name.startswith("total_load_actual") or name == "load_forecast_error_lag_24":
        return "historical_load"
    if "national" in name:
        return "weather_aggregate"
    if any(c.lower() in name for c in CITIES) or name.startswith("wind_deg"):
        return "historical_weather"
    if "generation" in name or name.startswith("renewable_share") or name.startswith("fossil_"):
        return "historical_generation"
    return "other"


def classify(name: str, group: str) -> str:
    if group == "calendar" or group == "historical_target" or group == "method_b":
        return "SAFE"
    if group == "day_ahead_forecast":
        return "UNKNOWN"
    if group in {"historical_load", "historical_generation", "historical_weather", "weather_aggregate"}:
        return "SAFE" if name.endswith("_168") or name.endswith("_48") else "FORBIDDEN"
    return "UNKNOWN"


audit = pd.DataFrame([{"feature": c, "group": group_of(c), "strict": classify(c, group_of(c))} for c in feat_plus])
print(audit["strict"].value_counts())
print("PRODUCTION_READY = FALSE")
print("STRICT y_pred = boş (kasıtlı)")
display(audit.groupby("strict").size().rename("n").to_frame())


## 12. Sonuç

- Ham CSV → temizlik → birleştirme → 184 SAFE özellik → kronolojik bölme
- Walk-forward ile Ridge α=0.001 ve METHOD_B (test yok)
- Kilitli testte Naive Lag-24’ü MAE’de geçer
- Geliştirme vs test sapma işareti farklı olabilir; rejim sorunu «çözüldü» denmez
- 24 saatlik operasyonel tahmin **üretime hazır değil**
- SHAP ilişki gösterir, nedensellik değil

Pano (isteğe bağlı, bu defterin parçası değil): `python3 -m streamlit run app.py`
